In [1]:
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [2]:
import catboost

In [3]:
df = pd.read_csv("Data/data_kern.csv")

In [4]:
df

,Calibration,Well,Sample,Depth,Stratigraphy,Mg,Al,Si,P,S,...,Sr,Zr,Mo,Ba,Pb,Th,U,Y,Cd,V
0,LE-FP,C3,1,2868.11,Fr,1.821148,7.515664,18.611483,0.299690,10.465318,...,0.017295,0.005169,0.006378,0.390684,0.0011,0.0034,0.0330,NaN,NaN,0.0
1,LE-FP,C3,2,2868.30,Fr,1.266357,7.461183,16.904798,0.000000,14.653735,...,0.009028,0.003092,0.007169,0.179474,0.0019,0.0000,0.0012,NaN,NaN,0.0
2,LE-FP,C3,3,2868.45,Fr,1.724880,7.150965,18.068731,0.000000,13.442425,...,0.018256,0.005030,0.006039,0.522104,0.0015,0.0000,0.0018,NaN,NaN,0.0
3,LE-FP,C3,4,2868.64,Fr,1.646439,8.260792,21.260196,0.000000,10.257598,...,0.011463,0.004061,0.005446,0.222917,0.0021,0.0000,0.0017,NaN,NaN,0.0
4,LE-FP,C3,5,2868.78,Fr,1.847176,7.757235,19.837340,0.000000,10.623499,...,0.016122,0.002446,0.006519,0.494492,0.0023,0.0000,0.0036,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3111,LE-FP,C7,362,2697.50,Tum,1.583924,10.335025,27.434503,0.029277,0.622963,...,0.012094,0.011722,0.000000,0.158503,NaN,NaN,NaN,NaN,NaN,NaN
3112,LE-FP,C7,363,2697.78,Tum,1.491458,7.860884,29.874495,0.020000,0.691707,...,0.011974,0.007845,0.000000,0.133802,NaN,NaN,NaN,NaN,NaN,NaN
3113,LE-FP,C7,364,2697.84,Tum,1.459012,11.210366,28.113407,0.052834,0.137574,...,0.012214,0.015968,0.000000,0.160868,NaN,NaN,NaN,NaN,NaN,NaN
3114,LE-FP,C7,365,2698.97,Tum,1.300348,1.489742,2.630132,0.000000,31.978827,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN


**Отобранные элементы с доставерными определениями**

In [5]:
elements = ['Mg', 'Al', 'Si', 'P', 'S', 'K', 'Ca', 'Ti', 'Mn', 'Fe', 'Ni', 'Cu', 'Zn', 'Se', 'Rb', 'Sr', 'Zr', 'Mo']

In [6]:
cols_to_keep = elements + ['Well', 'Sample',	'Depth',	'Stratigraphy']
df_filtered = df[cols_to_keep].query("Stratigraphy != 'Tum'").copy()

**Замена пачек на цифровые коды**

In [7]:
def to_str_key(val):
    try:
        return str(int(val))
    except (ValueError, TypeError):
        return str(val).strip()

df_filtered['Strat'] = df_filtered['Stratigraphy'].apply(to_str_key)

In [8]:
formation_map = {
    'Abl': 0,
    'A0': 1,
    '1': 2,
    '2a': 3,
    '2b': 4,
    '3': 5,
    '4a': 6,
    '4b': 7,
    '5a': 8,
    '5b': 9,
    '6': 10,
    'Fr': 11
}

df_filtered['Strat'] = df_filtered['Strat'].map(formation_map)

**Подсчет нулей по всем признакам**

In [9]:
# Подсчет нулей по всем признакам
zero_counts = df_filtered[elements].isnull().sum() + (df_filtered[elements] == 0).sum()

# Выводим результат
print("Количество нулевых значений по каждому элементу:")
print(zero_counts)

# Общее количество нулей
total_zeros = zero_counts.sum()
print(f"\nОбщее количество нулевых значений: {total_zeros}")

# Процент нулевых значений
total_values = len(df_filtered) * len(elements)
zero_percentage = (total_zeros / total_values) * 100
print(f"Процент нулевых значений: {zero_percentage:.2f}%")

Количество нулевых значений по каждому элементу:
Mg     392
Al       0
Si       3
P     1325
S        5
K      163
Ca     442
Ti       2
Mn     507
Fe     192
Ni     237
Cu     361
Zn      29
Se     378
Rb     278
Sr      93
Zr      28
Mo     598
dtype: int64

Общее количество нулевых значений: 5033
Процент нулевых значений: 9.02%


**Замена нулей на 2/3 LOD**

In [10]:
# LOD для каждого элемента
lod = {
    'Mg': 0.1848625,
    'Al': 0.0927875,
    'Si': 0.2984125,
    'Ca': 0.077025,
    'S': 0.3082,
    'K': 0.0271375,
    'Ti': 0.003,
    'Mn': 0.0024375,
    'Fe': 0.09165,
    'Ni': 0.0032625,
    'Cu': 0.00335,
    'Zn': 0.0022375,
    'Rb': 0.0003125,
    'Sr': 0.0013125,
    'Zr': 0.00065,
    'Mo': 0.00105,
    'P': 0.016325,
    'Se': 0.0002375
}

# Замена нулей на 2/3 LOD
for el in elements:
    replacement = (2/3) * lod[el]
    df_filtered[el] = df_filtered[el].replace(0, replacement)

# Проверка
print("Нулей после замены:")
print((df_filtered[elements] == 0).sum())

Нулей после замены:
Mg    0
Al    0
Si    0
P     0
S     0
K     0
Ca    0
Ti    0
Mn    0
Fe    0
Ni    0
Cu    0
Zn    0
Se    0
Rb    0
Sr    0
Zr    0
Mo    0
dtype: int64


**Создание новых признаков по соотношениям элементов**

In [11]:
# Карбонатность
df_filtered['(Ca+Mg)/(Al+Si)'] = (df_filtered['Ca'] + df_filtered['Mg']) / (df_filtered['Al'] + df_filtered['Si'])
df_filtered['Sr/Ca'] = df_filtered['Sr'] / df_filtered['Ca']
df_filtered['Mn/Ca'] = df_filtered['Mn'] / df_filtered['Ca']

# Терригенное вещество
df_filtered['(Al+K)/(Ca+Mg+Si)'] = (df_filtered['Al'] + df_filtered['K']) / (df_filtered['Ca'] + df_filtered['Mg'] + df_filtered['Si'])
df_filtered['Zr/Al'] = df_filtered['Zr'] / df_filtered['Al']
df_filtered['Zr/Si'] = df_filtered['Zr'] / df_filtered['Si']
df_filtered['K/Al'] = df_filtered['K'] / df_filtered['Al']
df_filtered['Ti/(Zr+Ti)'] = df_filtered['Ti'] / (df_filtered['Zr'] + df_filtered['Ti'])

# Морская биопродуктивность и кислородный режим
df_filtered['Si/Al'] = df_filtered['Si'] / df_filtered['Al']
df_filtered['P/Al'] = df_filtered['P'] / df_filtered['Al']
df_filtered['Ni/Fe'] = df_filtered['Ni'] / df_filtered['Fe']
df_filtered['Ni/(Si+Ca+Mg)'] = df_filtered['Ni'] / (df_filtered['Si'] + df_filtered['Ca'] + df_filtered['Mg'])
df_filtered['S/(Rb+Zr)'] = df_filtered['S'] / (df_filtered['Rb'] + df_filtered['Zr'])
df_filtered['Mo/Al'] = df_filtered['Mo'] / df_filtered['Al']
df_filtered['Mo/(Mn+Mo)'] = df_filtered['Mo'] / (df_filtered['Mn'] + df_filtered['Mo'])
df_filtered['Mn/Al'] = df_filtered['Mn'] / df_filtered['Al']

# Сульфидность
df_filtered['Fe/S'] = df_filtered['Fe'] / df_filtered['S']
df_filtered['Ni/S'] = df_filtered['Ni'] / df_filtered['S']

# Железистость
df_filtered['Fe/(Ca+Mg+Al)'] = df_filtered['Fe'] / (df_filtered['Ca'] + df_filtered['Mg'] + df_filtered['Al'])
df_filtered['Ti/(Si+Ca+Mg)'] = df_filtered['Ti'] / (df_filtered['Si'] + df_filtered['Ca'] + df_filtered['Mg'])
df_filtered['Fe/(Mn+Fe)'] = df_filtered['Fe'] / (df_filtered['Mn'] + df_filtered['Fe'])

In [13]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3099 entries, 0 to 3108
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Mg                 3099 non-null   float64
 1   Al                 3099 non-null   float64
 2   Si                 3099 non-null   float64
 3   P                  3099 non-null   float64
 4   S                  3099 non-null   float64
 5   K                  3099 non-null   float64
 6   Ca                 3099 non-null   float64
 7   Ti                 3099 non-null   float64
 8   Mn                 3099 non-null   float64
 9   Fe                 3099 non-null   float64
 10  Ni                 3099 non-null   float64
 11  Cu                 3099 non-null   float64
 12  Zn                 3099 non-null   float64
 13  Se                 3099 non-null   float64
 14  Rb                 3099 non-null   float64
 15  Sr                 3099 non-null   float64
 16  Zr                 3099 non-n

In [14]:
df_filtered.to_csv('Data/data_kern_ratios.csv', index=False)